In [1]:
!uv venv /content/.venv --python 3.12 --seed

!git clone https://github.com/eth-sri/diffusion-lm-watermark.git /content/diffusion-lm-watermark

!uv pip install -r /content/diffusion-lm-watermark/requirements.txt --python /content/.venv/bin/python --torch-backend=auto

!uv pip install -e /content/diffusion-lm-watermark --python /content/.venv/bin/python

zsh:1: command not found: uv
fatal : impossible de créer les répertoires de premier niveau dans '/content/diffusion-lm-watermark': Read-only file system
zsh:1: command not found: uv
zsh:1: command not found: uv


In [2]:
import json
import yaml
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")
DATA = REPO / "data/WaterBench"

WM_BATCHES = Path("/content/watermark_batches_wm")
WM_CONFIGS = Path("/content/production_configs_wm")

WM_BATCHES.mkdir(parents=True, exist_ok=True)
WM_CONFIGS.mkdir(parents=True, exist_ok=True)

datasets = {
    "2-1_longform_qa": {
        "source": "2-1_longform_qa.jsonl",
        "prompt": "You are a helpful assistant, please answer the following question within 300 words:\n{context}\n{input}"
    },
    "2-2_finance_qa": {
        "source": "2-2_finance_qa.jsonl",
        "prompt": "You are a helpful assistant, please answer the following question with financial knowledge within 300 words:\n{context}\n{input}"
    },
    "5-1_alpacafarm": {
        "source": "5-1_alpacafarm.jsonl",
        "prompt": "You are a helpful assistant, please answer the following instruction: \n{context}\n{input}"
    }
}

model_config = {
    "model_name": "GSAI-ML/LLaDA-8B-Instruct",
    "tokenizer_name": "GSAI-ML/LLaDA-8B-Instruct",
    "torch_dtype": "bfloat16",
    "device_map": "cuda",
    "trust_remote_code": True,
    "steps": 300,
    "gen_length": 300,
    "temperature": 0.5,
    "remasking": "random",
    "model_specific_arguments": {
        "cfg_scale": 0.0,
        "mask_id": 126336,
        "block_length": 25
    }
}

watermark_config = {
    "watermark_type": "DiffusionKGW_Optimal_Gaussian",
    "delta": 4.0,
    "convolution_kernel": [-1],
    "topk": 50,
    "seeding_scheme": "sumhash",
    "greenlist_type": "bernoulli",
    "greenlist_params": {"gamma": 0.25},
    "enforce_kl": False,
    "n_iter": 1
}

n_configs = 0

for prefix, info in datasets.items():
    source = DATA / info["source"]

    with source.open(encoding="utf-8") as f:
        prompts = [
            json.loads(line)
            for line in f
            if line.strip()
        ][:200]

    assert len(prompts) == 200, f"{prefix}: {len(prompts)} prompts"

    for batch_id in range(1, 11):
        batch_name = f"{prefix}_batch_{batch_id:02d}"
        batch_path = WM_BATCHES / f"{batch_name}.jsonl"

        selected = prompts[(batch_id - 1) * 20 : batch_id * 20]

        with batch_path.open("w", encoding="utf-8") as f:
            for row in selected:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

        config = {
            "model_configuration": model_config,
            "watermark_type": "Ours",
            "watermark_config": watermark_config,
            "evaluation_config": {
                "num_samples": 20,
                "batch_size": 1,
                "evaluation_datasets": [{
                    "path": str(batch_path),
                    "user_prompt": info["prompt"],
                    "chat": True,
                    "max_length": 150,
                    "min_length": 1
                }],
                "tqdm": True
            }
        }

        config_path = WM_CONFIGS / f"{batch_name}_watermarked.yaml"
        config_path.write_text(
            yaml.safe_dump(config, sort_keys=False),
            encoding="utf-8"
        )

        n_configs += 1

print(f"✅ {n_configs} configs watermarked créées")

✅ 30 configs watermarked créées


In [3]:
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")
PYTHON = "/content/.venv/bin/python"

WM_CONFIGS = Path("/content/production_configs_wm")
WM_RESULTS = Path("/content/production_results_wm")

WM_RESULTS.mkdir(parents=True, exist_ok=True)

remaining_batches = [
    f"{dataset}_batch_{i:02d}_watermarked"
    for dataset in ["2-2_finance_qa", "5-1_alpacafarm"]
    for i in range(1, 11)
]

for name in remaining_batches:
    assert (WM_CONFIGS / f"{name}.yaml").is_file(), name

print(f"✅ {len(remaining_batches)} batches prêts")
print("Finance QA : 10 batches")
print("AlpacaFarm : 10 batches")
print("Total : 400 réponses à générer")

✅ 20 batches prêts
Finance QA : 10 batches
AlpacaFarm : 10 batches
Total : 400 réponses à générer


In [4]:
import subprocess
import time
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")
PYTHON = "/content/.venv/bin/python"

WM_CONFIGS = Path("/content/production_configs_wm")
WM_RESULTS = Path("/content/production_results_wm")
WM_RESULTS.mkdir(parents=True, exist_ok=True)

batch_names = [
    f"2-2_finance_qa_batch_{i:02d}_watermarked"
    for i in range(1, 11)
]

start_all = time.time()

for idx, name in enumerate(batch_names, 1):
    config = WM_CONFIGS / f"{name}.yaml"
    output = WM_RESULTS / name
    result_file = output / "results.json"

    print("\n" + "=" * 70, flush=True)
    print(f"[{idx}/10] {name}", flush=True)

    assert config.is_file(), f"Configuration absente : {config}"

    if result_file.exists():
        print("⏭️ Déjà généré", flush=True)
        continue

    if output.exists():
        raise RuntimeError(f"Dossier déjà existant à vérifier : {output}")

    output.mkdir(parents=True)

    start_one = time.time()
    print("🚀 Génération en cours...", flush=True)

    process = subprocess.run(
        [
            PYTHON, "-u", "scripts/run_config.py",
            "--config", str(config),
            "--output_path", str(output),
            "--num_samples", "20"
        ],
        cwd=REPO
    )

    if process.returncode != 0:
        raise RuntimeError(f"Erreur sur {name}")

    if not result_file.exists():
        raise FileNotFoundError(result_file)

    print(f"✅ Terminé en {(time.time() - start_one)/60:.1f} min", flush=True)
    print(f"📊 Progression : {idx}/10 lots", flush=True)
    print(f"🕒 Temps total écoulé : {(time.time() - start_all)/60:.1f} min", flush=True)

print("\n🎉 Les 10 batches Finance QA watermarked sont terminés")


[1/10] 2-2_finance_qa_batch_01_watermarked
🚀 Génération en cours...
✅ Terminé en 7.8 min
📊 Progression : 1/10 lots
🕒 Temps total écoulé : 7.8 min

[2/10] 2-2_finance_qa_batch_02_watermarked
🚀 Génération en cours...

[4/10] 2-2_finance_qa_batch_04_watermarked
🚀 Génération en cours...
✅ Terminé en 6.6 min
📊 Progression : 4/10 lots
🕒 Temps total écoulé : 26.5 min

[5/10] 2-2_finance_qa_batch_05_watermarked
🚀 Génération en cours...
✅ Terminé en 6.1 min
📊 Progression : 5/10 lots
🕒 Temps total écoulé : 32.6 min

[6/10] 2-2_finance_qa_batch_06_watermarked
🚀 Génération en cours...
✅ Terminé en 6.1 min
📊 Progression : 6/10 lots
🕒 Temps total écoulé : 38.7 min

[7/10] 2-2_finance_qa_batch_07_watermarked
🚀 Génération en cours...


: 

In [5]:
from pathlib import Path
import json

base = Path("/content/production_results_wm")

for i in range(1, 11):
    name = f"2-2_finance_qa_batch_{i:02d}_watermarked"
    path = base / name / "results.json"

    if not path.exists():
        print(f"{name}: ABSENT")
        continue

    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    unique = len(set(r["prompt"] for r in rows))
    print(f"{name}: {len(rows)} lignes | {unique}/20 prompts uniques")

2-2_finance_qa_batch_01_watermarked: 20 lignes | 20/20 prompts uniques
2-2_finance_qa_batch_02_watermarked: 20 lignes | 20/20 prompts uniques
2-2_finance_qa_batch_03_watermarked: 20 lignes | 20/20 prompts uniques
2-2_finance_qa_batch_04_watermarked: 20 lignes | 20/20 prompts uniques
2-2_finance_qa_batch_05_watermarked: 20 lignes | 20/20 prompts uniques
2-2_finance_qa_batch_06_watermarked: 20 lignes | 20/20 prompts uniques
2-2_finance_qa_batch_07_watermarked: 20 lignes | 20/20 prompts uniques
2-2_finance_qa_batch_08_watermarked: 20 lignes | 20/20 prompts uniques
2-2_finance_qa_batch_09_watermarked: 20 lignes | 20/20 prompts uniques
2-2_finance_qa_batch_10_watermarked: 20 lignes | 20/20 prompts uniques


In [1]:
import base64
import html
from pathlib import Path
from IPython.display import HTML, display

base = Path("/content/production_results_wm")

for i in range(1, 11):
    name = f"2-2_finance_qa_batch_{i:02d}_watermarked"
    source = base / name / "results.json"

    if not source.is_file():
        print(f"❌ Fichier manquant : {name}")
        continue

    filename = f"{name}.jsonl"
    encoded = base64.b64encode(source.read_bytes()).decode("ascii")
    safe_name = html.escape(filename, quote=True)

    display(HTML(
        f'<a download="{safe_name}" '
        f'href="data:application/octet-stream;base64,{encoded}">'
        f'📥 Télécharger {safe_name}</a>'
    ))

print("✅ Liens de téléchargement créés")

❌ Fichier manquant : 2-2_finance_qa_batch_01_watermarked
❌ Fichier manquant : 2-2_finance_qa_batch_02_watermarked
❌ Fichier manquant : 2-2_finance_qa_batch_03_watermarked
❌ Fichier manquant : 2-2_finance_qa_batch_04_watermarked
❌ Fichier manquant : 2-2_finance_qa_batch_05_watermarked
❌ Fichier manquant : 2-2_finance_qa_batch_06_watermarked
❌ Fichier manquant : 2-2_finance_qa_batch_07_watermarked
❌ Fichier manquant : 2-2_finance_qa_batch_08_watermarked
❌ Fichier manquant : 2-2_finance_qa_batch_09_watermarked
❌ Fichier manquant : 2-2_finance_qa_batch_10_watermarked
✅ Liens de téléchargement créés


In [2]:
from pathlib import Path

files = list(Path("/content").rglob("2-2_finance_qa_batch_*_watermarked/results.json"))
print("Fichiers retrouvés :", len(files))
for f in files:
    print(f)

Fichiers retrouvés : 0


In [11]:
import json
import yaml
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")
CONFIGS = Path("/content/production_configs_wm")
BATCHES = Path("/content/watermark_batches_wm")

CONFIGS.mkdir(parents=True, exist_ok=True)
BATCHES.mkdir(parents=True, exist_ok=True)

source = REPO / "data/WaterBench/2-2_finance_qa.jsonl"

assert source.is_file(), f"Dataset introuvable : {source}"
assert Path("/content/.venv/bin/python").is_file(), "Environnement Python absent"

with source.open(encoding="utf-8") as f:
    prompts = [json.loads(line) for line in f if line.strip()][:100]

assert len(prompts) == 100

for i in range(1, 6):
    name = f"2-2_finance_qa_batch_{i:02d}"
    batch_path = BATCHES / f"{name}.jsonl"

    selected = prompts[(i - 1) * 20:i * 20]

    with batch_path.open("w", encoding="utf-8") as f:
        for row in selected:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    config = {
        "model_configuration": {
            "model_name": "GSAI-ML/LLaDA-8B-Instruct",
            "tokenizer_name": "GSAI-ML/LLaDA-8B-Instruct",
            "torch_dtype": "bfloat16",
            "device_map": "cuda",
            "trust_remote_code": True,
            "steps": 300,
            "gen_length": 300,
            "temperature": 0.5,
            "remasking": "random",
            "model_specific_arguments": {
                "cfg_scale": 0.0,
                "mask_id": 126336,
                "block_length": 25
            }
        },
        "watermark_type": "Ours",
        "watermark_config": {
            "watermark_type": "DiffusionKGW_Optimal_Gaussian",
            "delta": 4.0,
            "convolution_kernel": [-1],
            "topk": 50,
            "seeding_scheme": "sumhash",
            "greenlist_type": "bernoulli",
            "greenlist_params": {"gamma": 0.25},
            "enforce_kl": False,
            "n_iter": 1
        },
        "evaluation_config": {
            "num_samples": 20,
            "batch_size": 1,
            "evaluation_datasets": [{
                "path": str(batch_path),
                "user_prompt": (
                    "You are a helpful assistant, please answer the "
                    "following question with financial knowledge "
                    "within 300 words:\n{context}\n{input}"
                ),
                "chat": True,
                "max_length": 150,
                "min_length": 1
            }],
            "tqdm": True
        }
    }

    config_path = CONFIGS / f"{name}_watermarked.yaml"
    config_path.write_text(
        yaml.safe_dump(config, sort_keys=False),
        encoding="utf-8"
    )

    print(f"✅ {config_path.name}")

print("\n✅ Les 5 configurations sont prêtes.")

✅ 2-2_finance_qa_batch_01_watermarked.yaml
✅ 2-2_finance_qa_batch_02_watermarked.yaml
✅ 2-2_finance_qa_batch_03_watermarked.yaml
✅ 2-2_finance_qa_batch_04_watermarked.yaml
✅ 2-2_finance_qa_batch_05_watermarked.yaml

✅ Les 5 configurations sont prêtes.


In [12]:
import subprocess
import time
import json
import base64
import html
from pathlib import Path
from IPython.display import HTML, display

REPO = Path("/content/diffusion-lm-watermark")
PYTHON = "/content/.venv/bin/python"

CONFIGS = Path("/content/production_configs_wm")
RESULTS = Path("/content/production_results_wm")
RESULTS.mkdir(parents=True, exist_ok=True)

for i in range(1, 6):
    name = f"2-2_finance_qa_batch_{i:02d}_watermarked"
    config = CONFIGS / f"{name}.yaml"
    output = RESULTS / name
    result_file = output / "results.json"

    print(f"\n{'='*60}", flush=True)
    print(f"🚀 [{i}/5] {name}", flush=True)

    assert config.is_file(), f"Config manquante : {config}"

    if not result_file.exists():
        if output.exists():
            raise RuntimeError(
                f"Dossier incomplet existant : {output}. "
                "Ne pas écraser les données."
            )

        output.mkdir(parents=True)
        start = time.time()

        process = subprocess.run(
            [
                PYTHON, "-u", "scripts/run_config.py",
                "--config", str(config),
                "--output_path", str(output),
                "--num_samples", "20"
            ],
            cwd=REPO
        )

        if process.returncode != 0:
            raise RuntimeError(f"Erreur pendant {name}")

        print(f"⏱️ Durée : {(time.time()-start)/60:.1f} min", flush=True)

    # Vérifier les résultats
    assert result_file.is_file(), f"Résultats absents : {name}"

    rows = [
        json.loads(line)
        for line in result_file.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    unique = len({r["prompt"] for r in rows})

    print(f"✅ {len(rows)} réponses | {unique}/20 prompts uniques", flush=True)

    if len(rows) != 20 or unique != 20:
        raise RuntimeError(
            f"Résultats inattendus pour {name}. "
            "Fichier conservé, aucune régénération."
        )

    # Créer immédiatement le lien de téléchargement
    filename = f"{name}.jsonl"
    encoded = base64.b64encode(result_file.read_bytes()).decode("ascii")
    safe_name = html.escape(filename, quote=True)

    display(HTML(
        f'<a download="{safe_name}" '
        f'href="data:application/octet-stream;base64,{encoded}">'
        f'📥 Télécharger {safe_name}</a>'
    ))

    print(f"📊 Progression : {i}/5 batches terminés", flush=True)

print("\n🎉 Les 5 batches Finance QA sont terminés !")


🚀 [1/5] 2-2_finance_qa_batch_01_watermarked
⏱️ Durée : 8.2 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 1/5 batches terminés

🚀 [2/5] 2-2_finance_qa_batch_02_watermarked
⏱️ Durée : 6.1 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 2/5 batches terminés

🚀 [3/5] 2-2_finance_qa_batch_03_watermarked
⏱️ Durée : 6.4 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 3/5 batches terminés

🚀 [4/5] 2-2_finance_qa_batch_04_watermarked
⏱️ Durée : 7.3 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 4/5 batches terminés

🚀 [5/5] 2-2_finance_qa_batch_05_watermarked
⏱️ Durée : 7.0 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 5/5 batches terminés

🎉 Les 5 batches Finance QA sont terminés !


In [6]:
from pathlib import Path
import subprocess

repo = Path("/content/diffusion-lm-watermark")

if not (repo / ".git").exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/eth-sri/diffusion-lm-watermark.git",
        str(repo)
    ], check=True)

print("Dépôt :", "✅" if repo.exists() else "❌")

# Retrouver le vrai emplacement du dataset
matches = list(repo.rglob("2-2_finance_qa.jsonl"))

for p in matches:
    print("✅ Dataset trouvé :", p)

if not matches:
    print("❌ Dataset absent du dépôt")

Dépôt : ✅
✅ Dataset trouvé : /content/diffusion-lm-watermark/data/WaterBench/2-2_finance_qa.jsonl


SyntaxError: invalid character '✅' (U+2705) (3440894203.py, line 1)

In [13]:
import json
import yaml
from pathlib import Path

repo = Path("/content/diffusion-lm-watermark")
configs = Path("/content/production_configs_wm")
batches = Path("/content/watermark_batches_wm")

configs.mkdir(parents=True, exist_ok=True)
batches.mkdir(parents=True, exist_ok=True)

source = repo / "data/WaterBench/2-2_finance_qa.jsonl"

with source.open(encoding="utf-8") as f:
    prompts = [json.loads(line) for line in f if line.strip()][:200]

assert len(prompts) == 200

for i in range(6, 11):
    name = f"2-2_finance_qa_batch_{i:02d}"
    batch_path = batches / f"{name}.jsonl"

    with batch_path.open("w", encoding="utf-8") as f:
        for row in prompts[(i-1)*20:i*20]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    config = {
        "model_configuration": {
            "model_name": "GSAI-ML/LLaDA-8B-Instruct",
            "tokenizer_name": "GSAI-ML/LLaDA-8B-Instruct",
            "torch_dtype": "bfloat16",
            "device_map": "cuda",
            "trust_remote_code": True,
            "steps": 300,
            "gen_length": 300,
            "temperature": 0.5,
            "remasking": "random",
            "model_specific_arguments": {
                "cfg_scale": 0.0,
                "mask_id": 126336,
                "block_length": 25
            }
        },
        "watermark_type": "Ours",
        "watermark_config": {
            "watermark_type": "DiffusionKGW_Optimal_Gaussian",
            "delta": 4.0,
            "convolution_kernel": [-1],
            "topk": 50,
            "seeding_scheme": "sumhash",
            "greenlist_type": "bernoulli",
            "greenlist_params": {"gamma": 0.25},
            "enforce_kl": False,
            "n_iter": 1
        },
        "evaluation_config": {
            "num_samples": 20,
            "batch_size": 1,
            "evaluation_datasets": [{
                "path": str(batch_path),
                "user_prompt": (
                    "You are a helpful assistant, please answer the "
                    "following question with financial knowledge "
                    "within 300 words:\n{context}\n{input}"
                ),
                "chat": True,
                "max_length": 150,
                "min_length": 1
            }],
            "tqdm": True
        }
    }

    path = configs / f"{name}_watermarked.yaml"
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    print(f"✅ {path.name}")

print("✅ Les 5 configurations 06–10 sont prêtes !")

✅ 2-2_finance_qa_batch_06_watermarked.yaml
✅ 2-2_finance_qa_batch_07_watermarked.yaml
✅ 2-2_finance_qa_batch_08_watermarked.yaml
✅ 2-2_finance_qa_batch_09_watermarked.yaml
✅ 2-2_finance_qa_batch_10_watermarked.yaml
✅ Les 5 configurations 06–10 sont prêtes !


In [14]:
import subprocess
import time
import json
import base64
import html
from pathlib import Path
from IPython.display import HTML, display

REPO = Path("/content/diffusion-lm-watermark")
PYTHON = "/content/.venv/bin/python"

CONFIGS = Path("/content/production_configs_wm")
RESULTS = Path("/content/production_results_wm")
RESULTS.mkdir(parents=True, exist_ok=True)

for i in range(6, 11):
    name = f"2-2_finance_qa_batch_{i:02d}_watermarked"
    config = CONFIGS / f"{name}.yaml"
    output = RESULTS / name
    result_file = output / "results.json"

    print(f"\n{'='*60}", flush=True)
    print(f"🚀 [{i-5}/5] {name}", flush=True)

    assert config.is_file(), f"Config manquante : {config}"

    if not result_file.exists():
        if output.exists():
            raise RuntimeError(f"Dossier incomplet existant : {output}")

        output.mkdir(parents=True)
        start = time.time()

        process = subprocess.run(
            [
                PYTHON, "-u", "scripts/run_config.py",
                "--config", str(config),
                "--output_path", str(output),
                "--num_samples", "20"
            ],
            cwd=REPO
        )

        if process.returncode != 0:
            raise RuntimeError(f"Erreur pendant {name}")

        print(f"⏱️ Durée : {(time.time()-start)/60:.1f} min", flush=True)

    assert result_file.is_file(), f"Résultats absents : {name}"

    rows = [
        json.loads(line)
        for line in result_file.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    unique = len({r["prompt"] for r in rows})

    print(f"✅ {len(rows)} réponses | {unique}/20 prompts uniques", flush=True)

    if len(rows) != 20 or unique != 20:
        raise RuntimeError(f"Résultats inattendus pour {name}")

    filename = f"{name}.jsonl"
    encoded = base64.b64encode(result_file.read_bytes()).decode("ascii")
    safe_name = html.escape(filename, quote=True)

    display(HTML(
        f'<a download="{safe_name}" '
        f'href="data:application/octet-stream;base64,{encoded}">'
        f'📥 Télécharger {safe_name}</a>'
    ))

    print(f"📊 Progression : {i-5}/5 batches terminés", flush=True)

print("\n🎉 Les batches 06–10 sont terminés !")


🚀 [1/5] 2-2_finance_qa_batch_06_watermarked
⏱️ Durée : 6.7 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 1/5 batches terminés

🚀 [2/5] 2-2_finance_qa_batch_07_watermarked
⏱️ Durée : 6.2 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 2/5 batches terminés

🚀 [3/5] 2-2_finance_qa_batch_08_watermarked
⏱️ Durée : 6.4 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 3/5 batches terminés

🚀 [4/5] 2-2_finance_qa_batch_09_watermarked
⏱️ Durée : 6.2 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 4/5 batches terminés

🚀 [5/5] 2-2_finance_qa_batch_10_watermarked
⏱️ Durée : 6.7 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 5/5 batches terminés

🎉 Les batches 06–10 sont terminés !


In [15]:
import json
import yaml
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")
CONFIGS = Path("/content/production_configs_wm")
BATCHES = Path("/content/watermark_batches_wm")

CONFIGS.mkdir(parents=True, exist_ok=True)
BATCHES.mkdir(parents=True, exist_ok=True)

source = REPO / "data/WaterBench/5-1_alpacafarm.jsonl"
assert source.is_file(), f"Dataset introuvable : {source}"

with source.open(encoding="utf-8") as f:
    prompts = [json.loads(line) for line in f if line.strip()][:100]

assert len(prompts) == 100

for i in range(1, 6):
    name = f"5-1_alpacafarm_batch_{i:02d}"
    batch_path = BATCHES / f"{name}.jsonl"

    with batch_path.open("w", encoding="utf-8") as f:
        for row in prompts[(i-1)*20:i*20]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    config = {
        "model_configuration": {
            "model_name": "GSAI-ML/LLaDA-8B-Instruct",
            "tokenizer_name": "GSAI-ML/LLaDA-8B-Instruct",
            "torch_dtype": "bfloat16",
            "device_map": "cuda",
            "trust_remote_code": True,
            "steps": 300,
            "gen_length": 300,
            "temperature": 0.5,
            "remasking": "random",
            "model_specific_arguments": {
                "cfg_scale": 0.0,
                "mask_id": 126336,
                "block_length": 25
            }
        },
        "watermark_type": "Ours",
        "watermark_config": {
            "watermark_type": "DiffusionKGW_Optimal_Gaussian",
            "delta": 4.0,
            "convolution_kernel": [-1],
            "topk": 50,
            "seeding_scheme": "sumhash",
            "greenlist_type": "bernoulli",
            "greenlist_params": {"gamma": 0.25},
            "enforce_kl": False,
            "n_iter": 1
        },
        "evaluation_config": {
            "num_samples": 20,
            "batch_size": 1,
            "evaluation_datasets": [{
                "path": str(batch_path),
                "user_prompt": (
                    "You are a helpful assistant, please answer "
                    "the following instruction: \n{context}\n{input}"
                ),
                "chat": True,
                "max_length": 150,
                "min_length": 1
            }],
            "tqdm": True
        }
    }

    config_path = CONFIGS / f"{name}_watermarked.yaml"
    config_path.write_text(
        yaml.safe_dump(config, sort_keys=False),
        encoding="utf-8"
    )

    print(f"✅ {config_path.name}")

print("\n✅ Les 5 configurations AlpacaFarm sont prêtes !")

✅ 5-1_alpacafarm_batch_01_watermarked.yaml
✅ 5-1_alpacafarm_batch_02_watermarked.yaml
✅ 5-1_alpacafarm_batch_03_watermarked.yaml
✅ 5-1_alpacafarm_batch_04_watermarked.yaml
✅ 5-1_alpacafarm_batch_05_watermarked.yaml

✅ Les 5 configurations AlpacaFarm sont prêtes !


In [16]:
import subprocess
import time
import json
import base64
import html
from pathlib import Path
from IPython.display import HTML, display

REPO = Path("/content/diffusion-lm-watermark")
PYTHON = "/content/.venv/bin/python"
CONFIGS = Path("/content/production_configs_wm")
RESULTS = Path("/content/production_results_wm")

RESULTS.mkdir(parents=True, exist_ok=True)

for i in range(1, 6):
    name = f"5-1_alpacafarm_batch_{i:02d}_watermarked"
    config = CONFIGS / f"{name}.yaml"
    output = RESULTS / name
    result_file = output / "results.json"

    print(f"\n{'='*60}", flush=True)
    print(f"🚀 [{i}/5] {name}", flush=True)

    assert config.is_file(), f"Configuration manquante : {config}"

    if not result_file.exists():
        if output.exists():
            raise RuntimeError(
                f"Dossier incomplet existant : {output}"
            )

        output.mkdir(parents=True)
        start = time.time()

        process = subprocess.run(
            [
                PYTHON, "-u", "scripts/run_config.py",
                "--config", str(config),
                "--output_path", str(output),
                "--num_samples", "20"
            ],
            cwd=REPO
        )

        if process.returncode != 0:
            raise RuntimeError(f"Erreur pendant {name}")

        print(
            f"⏱️ Durée : {(time.time()-start)/60:.1f} min",
            flush=True
        )

    assert result_file.is_file(), f"Résultats absents : {name}"

    rows = [
        json.loads(line)
        for line in result_file.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    unique = len({r["prompt"] for r in rows})

    print(
        f"✅ {len(rows)} réponses | {unique}/20 prompts uniques",
        flush=True
    )

    if len(rows) != 20 or unique != 20:
        print("⚠️ Résultats inhabituels : fichier conservé en RAW")
        filename = f"{name}_RAW.jsonl"
    else:
        filename = f"{name}.jsonl"

    # Lien de téléchargement
    encoded = base64.b64encode(result_file.read_bytes()).decode("ascii")
    safe_name = html.escape(filename, quote=True)

    display(HTML(
        f'<a download="{safe_name}" '
        f'href="data:application/octet-stream;base64,{encoded}">'
        f'📥 Télécharger {safe_name}</a>'
    ))

    print(f"📊 Progression : {i}/5 batches", flush=True)

    if len(rows) != 20 or unique != 20:
        raise RuntimeError(
            f"Vérifier {name} avant de continuer. "
            "Ne pas régénérer automatiquement."
        )

print("\n🎉 Les 5 premiers batches AlpacaFarm sont terminés !")


🚀 [1/5] 5-1_alpacafarm_batch_01_watermarked
⏱️ Durée : 7.3 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 1/5 batches

🚀 [2/5] 5-1_alpacafarm_batch_02_watermarked
⏱️ Durée : 7.3 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 2/5 batches

🚀 [3/5] 5-1_alpacafarm_batch_03_watermarked
⏱️ Durée : 6.1 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 3/5 batches

🚀 [4/5] 5-1_alpacafarm_batch_04_watermarked
⏱️ Durée : 6.4 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 4/5 batches

🚀 [5/5] 5-1_alpacafarm_batch_05_watermarked
⏱️ Durée : 6.4 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 5/5 batches

🎉 Les 5 premiers batches AlpacaFarm sont terminés !


In [17]:
import json
import yaml
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")
CONFIGS = Path("/content/production_configs_wm")
BATCHES = Path("/content/watermark_batches_wm")

CONFIGS.mkdir(parents=True, exist_ok=True)
BATCHES.mkdir(parents=True, exist_ok=True)

source = REPO / "data/WaterBench/5-1_alpacafarm.jsonl"
assert source.is_file(), f"Dataset introuvable : {source}"

with source.open(encoding="utf-8") as f:
    prompts = [json.loads(line) for line in f if line.strip()][:200]

assert len(prompts) == 200

for i in range(6, 11):
    name = f"5-1_alpacafarm_batch_{i:02d}"
    batch_path = BATCHES / f"{name}.jsonl"

    with batch_path.open("w", encoding="utf-8") as f:
        for row in prompts[(i-1)*20:i*20]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    config = {
        "model_configuration": {
            "model_name": "GSAI-ML/LLaDA-8B-Instruct",
            "tokenizer_name": "GSAI-ML/LLaDA-8B-Instruct",
            "torch_dtype": "bfloat16",
            "device_map": "cuda",
            "trust_remote_code": True,
            "steps": 300,
            "gen_length": 300,
            "temperature": 0.5,
            "remasking": "random",
            "model_specific_arguments": {
                "cfg_scale": 0.0,
                "mask_id": 126336,
                "block_length": 25
            }
        },
        "watermark_type": "Ours",
        "watermark_config": {
            "watermark_type": "DiffusionKGW_Optimal_Gaussian",
            "delta": 4.0,
            "convolution_kernel": [-1],
            "topk": 50,
            "seeding_scheme": "sumhash",
            "greenlist_type": "bernoulli",
            "greenlist_params": {"gamma": 0.25},
            "enforce_kl": False,
            "n_iter": 1
        },
        "evaluation_config": {
            "num_samples": 20,
            "batch_size": 1,
            "evaluation_datasets": [{
                "path": str(batch_path),
                "user_prompt": (
                    "You are a helpful assistant, please answer "
                    "the following instruction: \n{context}\n{input}"
                ),
                "chat": True,
                "max_length": 150,
                "min_length": 1
            }],
            "tqdm": True
        }
    }

    config_path = CONFIGS / f"{name}_watermarked.yaml"
    config_path.write_text(
        yaml.safe_dump(config, sort_keys=False),
        encoding="utf-8"
    )

    print(f"✅ {config_path.name}")

print("\n✅ Les 5 dernières configurations sont prêtes !")

✅ 5-1_alpacafarm_batch_06_watermarked.yaml
✅ 5-1_alpacafarm_batch_07_watermarked.yaml
✅ 5-1_alpacafarm_batch_08_watermarked.yaml
✅ 5-1_alpacafarm_batch_09_watermarked.yaml
✅ 5-1_alpacafarm_batch_10_watermarked.yaml

✅ Les 5 dernières configurations sont prêtes !


In [18]:
import subprocess
import time
import json
import base64
import html
from pathlib import Path
from IPython.display import HTML, display

REPO = Path("/content/diffusion-lm-watermark")
PYTHON = "/content/.venv/bin/python"
CONFIGS = Path("/content/production_configs_wm")
RESULTS = Path("/content/production_results_wm")

RESULTS.mkdir(parents=True, exist_ok=True)

for i in range(6, 11):
    name = f"5-1_alpacafarm_batch_{i:02d}_watermarked"
    config = CONFIGS / f"{name}.yaml"
    output = RESULTS / name
    result_file = output / "results.json"

    print(f"\n{'='*60}", flush=True)
    print(f"🚀 [{i-5}/5] {name}", flush=True)

    assert config.is_file(), f"Configuration manquante : {config}"

    if not result_file.exists():
        if output.exists():
            raise RuntimeError(
                f"Dossier incomplet existant : {output}"
            )

        output.mkdir(parents=True)
        start = time.time()

        process = subprocess.run(
            [
                PYTHON, "-u", "scripts/run_config.py",
                "--config", str(config),
                "--output_path", str(output),
                "--num_samples", "20"
            ],
            cwd=REPO
        )

        if process.returncode != 0:
            raise RuntimeError(f"Erreur pendant {name}")

        print(
            f"⏱️ Durée : {(time.time()-start)/60:.1f} min",
            flush=True
        )

    assert result_file.is_file(), f"Résultats absents : {name}"

    rows = [
        json.loads(line)
        for line in result_file.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    unique = len({r["prompt"] for r in rows})

    print(
        f"✅ {len(rows)} réponses | {unique}/20 prompts uniques",
        flush=True
    )

    filename = (
        f"{name}.jsonl"
        if len(rows) == 20 and unique == 20
        else f"{name}_RAW.jsonl"
    )

    encoded = base64.b64encode(result_file.read_bytes()).decode("ascii")
    safe_name = html.escape(filename, quote=True)

    display(HTML(
        f'<a download="{safe_name}" '
        f'href="data:application/octet-stream;base64,{encoded}">'
        f'📥 Télécharger {safe_name}</a>'
    ))

    print(f"📊 Progression : {i-5}/5 batches", flush=True)

    if len(rows) != 20 or unique != 20:
        print("⚠️ Résultats inhabituels. Fichier RAW à examiner.")
        break

print("\n✅ Fin de la génération AlpacaFarm 06–10")


🚀 [1/5] 5-1_alpacafarm_batch_06_watermarked
⏱️ Durée : 6.7 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 1/5 batches

🚀 [2/5] 5-1_alpacafarm_batch_07_watermarked
⏱️ Durée : 8.3 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 2/5 batches

🚀 [3/5] 5-1_alpacafarm_batch_08_watermarked
⏱️ Durée : 8.5 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 3/5 batches

🚀 [4/5] 5-1_alpacafarm_batch_09_watermarked
⏱️ Durée : 7.8 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 4/5 batches

🚀 [5/5] 5-1_alpacafarm_batch_10_watermarked
⏱️ Durée : 9.5 min
✅ 20 réponses | 20/20 prompts uniques


📊 Progression : 5/5 batches

✅ Fin de la génération AlpacaFarm 06–10
